In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, RocCurveDisplay)
from sklearn.model_selection import GridSearchCV
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

# Rutas
PROCESSED = '../data/processed/'

# Cargamos features
df = pd.read_pickle(PROCESSED + 'df_features.pkl')

# Separamos features y target
X = df.drop(columns=['success'])
y = df['success']

# Split estratificado — 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # garantiza mismo balance en train y test
)

# Verificación
print("Train:", X_train.shape, "| Balance:", y_train.mean().round(3))
print("Test: ", X_test.shape,  "| Balance:", y_test.mean().round(3))

Train: (2583, 28) | Balance: 0.755
Test:  (646, 28) | Balance: 0.755


In [2]:
# Aplicamos SMOTE solo sobre train
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

# Verificación
print("Train original:")
print(f"  Éxitos: {y_train.sum()} | Fracasos: {(y_train==0).sum()}")
print(f"  Balance: {y_train.mean().round(3)}")

print("\nTrain tras SMOTE:")
print(f"  Éxitos: {y_train_bal.sum()} | Fracasos: {(y_train_bal==0).sum()}")
print(f"  Balance: {y_train_bal.mean().round(3)}")

print("\nTest (sin modificar):")
print(f"  Éxitos: {y_test.sum()} | Fracasos: {(y_test==0).sum()}")
print(f"  Balance: {y_test.mean().round(3)}")

Train original:
  Éxitos: 1950 | Fracasos: 633
  Balance: 0.755

Train tras SMOTE:
  Éxitos: 1950 | Fracasos: 1950
  Balance: 0.5

Test (sin modificar):
  Éxitos: 488 | Fracasos: 158
  Balance: 0.755


In [4]:
# Verificación escalado
print("Budget train — media:", round(X_train_bal_scaled['budget'].mean(), 4),
      "| std:", round(X_train_bal_scaled['budget'].std(), 4))
print("Budget test  — media:", round(X_test_scaled['budget'].mean(), 4),
      "| std:", round(X_test_scaled['budget'].std(), 4))

# Logistic Regression
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_bal_scaled, y_train_bal)

# Predicciones
y_pred_lr = lr.predict(X_test_scaled)
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]

# Métricas
print("\n--- LOGISTIC REGRESSION ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_lr):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_lr):.4f}")
print(f"F1:        {f1_score(y_test, y_pred_lr):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob_lr):.4f}")
print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred_lr))

Budget train — media: 0.0 | std: 1.0001
Budget test  — media: 0.089 | std: 1.2066

--- LOGISTIC REGRESSION ---
Accuracy:  0.7167
Precision: 0.7861
Recall:    0.8586
F1:        0.8208
ROC-AUC:   0.6553

Matriz de confusión:
[[ 44 114]
 [ 69 419]]


In [5]:
# Definimos el grid de hiperparámetros a explorar
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5]
}

# Cross-validation estratificada — respeta el balance en cada fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# GridSearchCV — optimizamos por ROC-AUC, la métrica más honesta
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

print("Entrenando GridSearchCV — puede tardar unos minutos...")
grid_search.fit(X_train_bal_scaled, y_train_bal)

print("\nMejores hiperparámetros:", grid_search.best_params_)
print("Mejor ROC-AUC en CV:", round(grid_search.best_score_, 4))

Entrenando GridSearchCV — puede tardar unos minutos...
Fitting 5 folds for each of 12 candidates, totalling 60 fits

Mejores hiperparámetros: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}
Mejor ROC-AUC en CV: 0.9204
